# Benchmark 04: Max Token Sweep

**Goal:** Measure how response time changes as we increase `max_tokens`.

**Setup:** Ollama running locally with llama3.2

**What we're measuring:**
- TTFT (Time to First Token) — how long until the first token appears
- Total response time per request
- How output length affects decode time

**Why this matters:** TTFT and total generation time are different bottlenecks. This benchmark shows how a longer completion changes user-visible latency and overall throughput.

## Setup — make sure Ollama is running

Before running this notebook:
```bash
ollama serve
ollama pull llama3.2
```

In [8]:
import requests
import time
import json
import threading
import statistics

from typer import prompt

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3.2"
PROMPT = "Explain what a KV cache is in two sentences."
MAX_TOKENS = 128

def single_request(prompt=PROMPT, stream=False, max_tokens=MAX_TOKENS):
    """Send one request and measure total time and TTFT."""
    # After the response, check how many tokens were actually generated
    payload = {"model": MODEL, "prompt": prompt, "stream": stream, "options": {"num_predict": max_tokens}}
    
    start = time.time()
    
    if stream:
        # Streaming: capture time to first token
        first_token_time = None
        full_response = ""
        with requests.post(OLLAMA_URL, json=payload, stream=True) as r:
            for line in r.iter_lines():
                if line:
                    data = json.loads(line)
                    if first_token_time is None and data.get("response"):
                        first_token_time = time.time() - start
                    full_response += data.get("response", "")
                    if data.get("done"):
                        break
        total_time = time.time() - start
        return {
            "ttft": first_token_time,
            "total_time": total_time,
            "response": full_response
        }
    else:
        r = requests.post(OLLAMA_URL, json=payload)
        total_time = time.time() - start
        return {
            "ttft": total_time,  # non-streaming: entire response at once
            "total_time": total_time,
            "response": r.json().get("response", "")[:100]
        }

print("Functions loaded. Ready to benchmark.")

Functions loaded. Ready to benchmark.


## Experiment Max token sweep

In [9]:
import time

prompts = {
    128: "Explain KV cache in one sentence.",
    512: "Explain KV cache in detail covering memory layout, why it exists, and how it affects inference speed.",
    2048: "Write a comprehensive technical explanation of KV cache covering: what it is, the memory formula, why it grows with context length, how PagedAttention addresses it, and the production implications for serving many concurrent users."
}

for max_tokens, prompt in prompts.items():
    result = single_request(prompt=prompt, stream=True, max_tokens=max_tokens)
    print(f"max_tokens={max_tokens}")
    print(f"  TTFT:          {result['ttft']:.2f}s")
    print(f"  Total time:    {result['total_time']:.2f}s")
    print(f"  Response chars: {len(result['response'])}")
    print(f"  Preview: {result['response'][:80]}")
    print()

max_tokens=128
  TTFT:          2.93s
  Total time:    6.36s
  Response chars: 219
  Preview: A Key-Value (KV) cache is a type of memory caching system that stores frequently

max_tokens=512
  TTFT:          3.33s
  Total time:    55.75s
  Response chars: 2740
  Preview: KV Cache is a memory caching mechanism used in deep learning models to store int

max_tokens=2048
  TTFT:          4.35s
  Total time:    87.79s
  Response chars: 3951
  Preview: **What is KV Cache?**

KV Cache (Key-Value Cache) is a caching layer that stores



## What these numbers mean

**If per-request latency stays roughly flat** as concurrency increases:
- The server is handling requests in parallel
- Good news for throughput

**If per-request latency increases linearly** (2x at 2 concurrent, 4x at 4 concurrent):
- Requests are being queued and served one at a time
- This is Ollama's default behavior — it's a local dev tool, not a production server
- This is exactly the problem vLLM's continuous batching solves

**The insight:**  
TTFT stayed nearly flat — 2.93s, 3.33s, 4.35s — even as total time exploded from 6s to 87s.  
That means two separate things are happening:
- **Prefill phase:** the model reads the prompt and prepares generation. This is parallel and happens once, so TTFT stays relatively stable.
- **Decode phase:** the model generates tokens one by one. This is sequential, so total time rises quickly as `max_tokens` increases.

**Why this matters in production:**
- Chatbots care most about TTFT, because users notice when nothing appears on screen. Streaming helps here.
- Batch document jobs care more about total throughput, because the full response matters more than the first token.

The numbers here show the split clearly: 128 tokens had TTFT 2.93s and decode time 3.43s, while 2048 tokens had TTFT 4.35s and decode time 83.44s.